In [1]:
import pandas as pd

df = pd.read_csv("../data/youtoxic_english_1000.csv")
df = df.drop_duplicates(subset='Text', keep='first').reset_index(drop=True)

print("Filas:", df.shape[0])

Filas: 997


In [2]:
# Comprobación de valores nulos
print(df.isnull().sum())  

CommentId          0
VideoId            0
Text               0
IsToxic            0
IsAbusive          0
IsThreat           0
IsProvocative      0
IsObscene          0
IsHatespeech       0
IsRacist           0
IsNationalist      0
IsSexist           0
IsHomophobic       0
IsReligiousHate    0
IsRadicalism       0
dtype: int64


In [3]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# Keep negations and intensifiers, since they carry meaning
# that is critical for hate speech detection
negations_and_intensifiers = {
    'no', 'not', 'nor', 'never', 'none', 'nothing', 'nowhere',
    'neither', 'very', 'too', 'so', 'only', 'just'
}

stop_words = set(stopwords.words('english')) - negations_and_intensifiers
lemmatizer = WordNetLemmatizer()

def clean_text(text: str) -> str:
    """
    Clean raw text for NLP processing:
    - Lowercase
    - Remove URLs, mentions, and special characters
    - Remove stopwords (excluding negations/intensifiers, which
      are meaningful for hate speech detection)
    - Lemmatize tokens
    """
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words]
    return " ".join(tokens)

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Coder\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Coder\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Coder\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [4]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')

# Keep negations and intensifiers, since they carry meaning
# that is critical for hate speech detection
negations_and_intensifiers = {
    'no', 'not', 'nor', 'never', 'none', 'nothing', 'nowhere',
    'neither', 'very', 'too', 'so', 'only', 'just'
}

stop_words = set(stopwords.words('english')) - negations_and_intensifiers
lemmatizer = WordNetLemmatizer()

def clean_text(text: str) -> str:
    """
    Clean raw text for NLP processing:
    - Lowercase
    - Remove URLs, mentions, and special characters
    - Remove stopwords (excluding negations/intensifiers, which
      are meaningful for hate speech detection)
    - Lemmatize tokens
    """
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"@\w+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    tokens = text.split()
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words]
    return " ".join(tokens)

df['clean_text'] = df['Text'].apply(clean_text)
df[['Text', 'clean_text']].head()

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Coder\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Coder\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Coder\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


,Text,clean_text
0,If only people would just take a step back and...,only people would just take step back not make...
1,Law enforcement is not trained to shoot to app...,law enforcement not trained shoot apprehend tr...
2,\nDont you reckon them 'black lives matter' ba...,dont reckon black life matter banner held whit...
3,There are a very large number of people who do...,very large number people not like police offic...
4,"The Arab dude is absolutely right, he should h...",arab dude absolutely right not shot extra time...


In [5]:
# Comprobar si algún texto quedó vacío tras la limpieza 
empty_after_cleaning = df[df['clean_text'].str.strip() == '']
print("Filas vacías tras la limpieza:", len(empty_after_cleaning))


Filas vacías tras la limpieza: 0


In [6]:
df['clean_text_len'] =df['clean_text'].apply(lambda x: len(x.split()))
df['clean_text_len'].describe()

count    997.000000
mean      18.137412
std       25.151193
min        1.000000
25%        5.000000
50%       11.000000
75%       21.000000
max      406.000000
Name: clean_text_len, dtype: float64

In [7]:
import nltk
from nltk.stem import WordNetLemmatizer
from nltk.tokenize import word_tokenize

# Descarga de recursos necesarios (solo la primera vez)
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('wordnet')
nltk.download('omw-1.4')

lemmatizer = WordNetLemmatizer()

def tokenize_and_lemmatize(text):
    """
    Tokeniza el texto y aplica lematización a cada token.
    """
    tokens = word_tokenize(text)
    lemmatized_tokens = [lemmatizer.lemmatize(token) for token in tokens]
    return lemmatized_tokens

# Aplicamos la función sobre clean_text
df['tokens'] = df['clean_text'].apply(tokenize_and_lemmatize)

# Vista rápida del resultado
df[['clean_text', 'tokens']].head()

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Coder\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\Coder\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Coder\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     C:\Users\Coder\AppData\Roaming\nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


,clean_text,tokens
0,only people would just take step back not make...,"[only, people, would, just, take, step, back, ..."
1,law enforcement not trained shoot apprehend tr...,"[law, enforcement, not, trained, shoot, appreh..."
2,dont reckon black life matter banner held whit...,"[dont, reckon, black, life, matter, banner, he..."
3,very large number people not like police offic...,"[very, large, number, people, not, like, polic..."
4,arab dude absolutely right not shot extra time...,"[arab, dude, absolutely, right, not, shot, ext..."


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer

# TfidfVectorizer espera strings, no listas de tokens, así que los unimos de nuevo
df['tokens_text'] = df['tokens'].apply(lambda tokens: ' '.join(tokens))

tfidf = TfidfVectorizer(
    max_features=1000,   # antes 5000 — reduce el vocabulario para evitar sobreajuste
    ngram_range=(1, 1),  # solo unigramas, sin bigramas (menos ruido/variables)
    min_df=5             # ignora palabras que aparecen en menos de 5 comentarios (antes 2)
)

X_tfidf = tfidf.fit_transform(df['tokens_text'])

print(f"Forma de la matriz TF-IDF: {X_tfidf.shape}")
print(f"Vocabulario (primeras 20 palabras): {list(tfidf.get_feature_names_out()[:20])}")

Forma de la matriz TF-IDF: (997, 701)
Vocabulario (primeras 20 palabras): ['able', 'absolutely', 'account', 'act', 'action', 'actual', 'actually', 'african', 'agree', 'air', 'al', 'allowed', 'almost', 'already', 'also', 'always', 'amen', 'america', 'american', 'animal']


In [9]:
from sklearn.model_selection import train_test_split

# Definimos X (features) e y (target)
X = X_tfidf
y = df['IsToxic']

# Split train/test (80/20 es un buen punto de partida)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,      # para que el split sea reproducible
    stratify=y             # mantiene la misma proporción de tóxico/no tóxico en train y test
)

print(f"Train: {X_train.shape[0]} comentarios")
print(f"Test: {X_test.shape[0]} comentarios")
print(f"\nDistribución en train:\n{y_train.value_counts(normalize=True)}")
print(f"\nDistribución en test:\n{y_test.value_counts(normalize=True)}")

Train: 797 comentarios
Test: 200 comentarios

Distribución en train:
IsToxic
False    0.539523
True     0.460477
Name: proportion, dtype: float64

Distribución en test:
IsToxic
False    0.54
True     0.46
Name: proportion, dtype: float64


In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, confusion_matrix

# --- Modelo 1: Logistic Regression ---
log_reg = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
log_reg.fit(X_train, y_train)
y_pred_lr = log_reg.predict(X_test)

print("=== Logistic Regression ===")
print(classification_report(y_test, y_pred_lr))

# --- Modelo 2: Naive Bayes ---
nb = MultinomialNB()
nb.fit(X_train, y_train)
y_pred_nb = nb.predict(X_test)

print("\n=== Naive Bayes ===")
print(classification_report(y_test, y_pred_nb))

=== Logistic Regression ===
              precision    recall  f1-score   support

       False       0.68      0.75      0.71       108
        True       0.67      0.59      0.62        92

    accuracy                           0.68       200
   macro avg       0.67      0.67      0.67       200
weighted avg       0.67      0.68      0.67       200


=== Naive Bayes ===
              precision    recall  f1-score   support

       False       0.70      0.81      0.75       108
        True       0.72      0.60      0.65        92

    accuracy                           0.71       200
   macro avg       0.71      0.70      0.70       200
weighted avg       0.71      0.71      0.71       200



In [11]:
from sklearn.svm import LinearSVC

svm = LinearSVC(class_weight='balanced', random_state=42, max_iter=2000)
svm.fit(X_train, y_train)
y_pred_svm = svm.predict(X_test)

print("=== SVM (LinearSVC) ===")
print(classification_report(y_test, y_pred_svm))

=== SVM (LinearSVC) ===
              precision    recall  f1-score   support

       False       0.69      0.75      0.72       108
        True       0.67      0.61      0.64        92

    accuracy                           0.69       200
   macro avg       0.68      0.68      0.68       200
weighted avg       0.68      0.69      0.68       200



In [12]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("=== Random Forest ===")
print(classification_report(y_test, y_pred_rf))

=== Random Forest ===
              precision    recall  f1-score   support

       False       0.71      0.84      0.77       108
        True       0.76      0.60      0.67        92

    accuracy                           0.73       200
   macro avg       0.74      0.72      0.72       200
weighted avg       0.74      0.73      0.72       200



In [13]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [3, 4, 5, 6],
    'min_samples_split': [15, 20, 30],
    'min_samples_leaf': [5, 10, 15],
    'max_features': ['sqrt', 'log2'],
}

rf_grid = GridSearchCV(
    RandomForestClassifier(class_weight='balanced', random_state=42),
    param_grid,
    cv=5,
    scoring='f1',   # optimizamos F1, no accuracy, porque nos importa la clase tóxico
    n_jobs=-1       # usa todos los núcleos disponibles para ir más rápido
)

rf_grid.fit(X_train, y_train)

print("Mejores parámetros:", rf_grid.best_params_)
print("Mejor F1 (cross-validation):", rf_grid.best_score_)

best_rf = rf_grid.best_estimator_
y_pred_best = best_rf.predict(X_test)
print("\n=== Random Forest optimizado ===")
print(classification_report(y_test, y_pred_best))

Mejores parámetros: {'max_depth': 3, 'max_features': 'log2', 'min_samples_leaf': 5, 'min_samples_split': 15, 'n_estimators': 100}
Mejor F1 (cross-validation): 0.6574545803934952

=== Random Forest optimizado ===
              precision    recall  f1-score   support

       False       0.65      0.83      0.73       108
        True       0.71      0.48      0.57        92

    accuracy                           0.67       200
   macro avg       0.68      0.66      0.65       200
weighted avg       0.68      0.67      0.66       200



In [14]:
from sklearn.metrics import f1_score

# Comprobación de overfitting para los 4 modelos entrenados
modelos = {
    "Logistic Regression": log_reg,
    "Naive Bayes": nb,
    "SVM": svm,
    "Random Forest (optimizado)": best_rf,
}

print(f"{'Modelo':<28} {'F1 train':>10} {'F1 test':>10} {'Diferencia':>12}")
for nombre, modelo in modelos.items():
    f1_train = f1_score(y_train, modelo.predict(X_train))
    f1_test = f1_score(y_test, modelo.predict(X_test))
    diff = (f1_train - f1_test) * 100
    print(f"{nombre:<28} {f1_train:>10.4f} {f1_test:>10.4f} {diff:>10.2f} pts")

Modelo                         F1 train    F1 test   Diferencia
Logistic Regression              0.8834     0.6243      25.91 pts
Naive Bayes                      0.8559     0.6548      20.12 pts
SVM                              0.9507     0.6400      31.07 pts
Random Forest (optimizado)       0.7622     0.5714      19.08 pts


In [15]:
from sklearn.model_selection import cross_val_score

print(f"{'Modelo':<28} {'F1 CV (5-fold)':>15} {'F1 test':>10} {'Diferencia':>12}")
for nombre, modelo in modelos.items():
    cv_scores = cross_val_score(modelo, X_train, y_train, cv=5, scoring='f1')
    f1_cv = cv_scores.mean()
    f1_test = f1_score(y_test, modelo.predict(X_test))
    diff = (f1_cv - f1_test) * 100
    print(f"{nombre:<28} {f1_cv:>15.4f} {f1_test:>10.4f} {diff:>10.2f} pts")

Modelo                        F1 CV (5-fold)    F1 test   Diferencia


Logistic Regression                   0.6739     0.6243       4.96 pts
Naive Bayes                           0.6538     0.6548      -0.09 pts
SVM                                   0.6471     0.6400       0.71 pts
Random Forest (optimizado)            0.6575     0.5714       8.60 pts


In [16]:
from sklearn.ensemble import VotingClassifier

# Ensemble por votación mayoritaria (hard voting) de los 3 modelos con buen balance
# rendimiento/generalización. Random Forest queda fuera: aun optimizado, tenía
# el peor overfitting (8.60 pts en CV vs test) y el peor F1 test (0.57).
ensemble = VotingClassifier(
    estimators=[
        ('log_reg', log_reg),
        ('naive_bayes', nb),
        ('svm', svm),
    ],
    voting='hard'
)
ensemble.fit(X_train, y_train)
y_pred_ensemble = ensemble.predict(X_test)

print("=== Ensemble (Voting Classifier) ===")
print(classification_report(y_test, y_pred_ensemble))

=== Ensemble (Voting Classifier) ===
              precision    recall  f1-score   support

       False       0.69      0.75      0.72       108
        True       0.67      0.60      0.63        92

    accuracy                           0.68       200
   macro avg       0.68      0.67      0.67       200
weighted avg       0.68      0.68      0.68       200



In [17]:
# Comprobación de overfitting del ensemble (CV vs test, igual que con los modelos individuales)
cv_scores_ensemble = cross_val_score(ensemble, X_train, y_train, cv=5, scoring='f1')
f1_cv_ensemble = cv_scores_ensemble.mean()
f1_test_ensemble = f1_score(y_test, y_pred_ensemble)
diff_ensemble = (f1_cv_ensemble - f1_test_ensemble) * 100

print(f"F1 CV (5-fold): {f1_cv_ensemble:.4f} | F1 test: {f1_test_ensemble:.4f} | Diferencia: {diff_ensemble:.2f} pts")

F1 CV (5-fold): 0.6786 | F1 test: 0.6322 | Diferencia: 4.64 pts


In [18]:
import joblib

joblib.dump(ensemble, '../models/ensemble_toxic_model.pkl')
joblib.dump(tfidf, '../models/tfidf_vectorizer.pkl')

print("Modelo y vectorizador guardados correctamente.")

Modelo y vectorizador guardados correctamente.


In [19]:
def predecir_toxicidad(comentario):
    texto_limpio = clean_text(comentario)
    tokens = tokenize_and_lemmatize(texto_limpio)
    tokens_text = ' '.join(tokens)
    vector = tfidf.transform([tokens_text])
    prediccion = ensemble.predict(vector)[0]
    return "Tóxico" if prediccion else "No tóxico"
    

# Comentarios de prueba (sustituye por comentarios reales que copies de YouTube)
comentarios_prueba = [
    "This video was really helpful, thank you for explaining it so clearly!",
    "You are so stupid, how can anyone believe this garbage",
    "I disagree with some points but overall good content",
]

for comentario in comentarios_prueba:
    resultado = predecir_toxicidad(comentario)
    print(f"'{comentario[:50]}...' -> {resultado}")

'This video was really helpful, thank you for expla...' -> No tóxico
'You are so stupid, how can anyone believe this gar...' -> Tóxico
'I disagree with some points but overall good conte...' -> No tóxico


In [20]:
# Comentarios reales de YouTube (fuente: Sage Journals)
comentarios_prueba = [
    # Tóxicos
    "She is a brat.",
    "kick her pompous ass",
    "What a spoiled child",
    "Go back to school, little bitch.",
    "Fuck off to school.",
    # No tóxicos / neutros
    "Goodbye to my home in Bahrain",
    "RIP Florida",
    "Australia is covered by sea, but Melbourne is totally safe",
    "How can we stop this from happening",
    "I wouldnt mind if humans got extinct to be honest.",
]

for comentario in comentarios_prueba:
    resultado = predecir_toxicidad(comentario)
    print(f"'{comentario[:50]}...' -> {resultado}")


'She is a brat....' -> No tóxico
'kick her pompous ass...' -> No tóxico
'What a spoiled child...' -> No tóxico
'Go back to school, little bitch....' -> Tóxico
'Fuck off to school....' -> Tóxico
'Goodbye to my home in Bahrain...' -> No tóxico
'RIP Florida...' -> No tóxico
'Australia is covered by sea, but Melbourne is tota...' -> No tóxico
'How can we stop this from happening...' -> No tóxico
'I wouldnt mind if humans got extinct to be honest....' -> No tóxico


## Conclusiones del preprocesamiento y modelado

- Limpieza de texto: minúsculas, eliminación de URLs/menciones/caracteres especiales,
  stopwords personalizadas (se conservan negaciones e intensificadores como "not",
  "never", "very" por ser relevantes para detectar discurso de odio).
- Tokenización y lematización con NLTK (WordNetLemmatizer) sobre el texto limpio.
- Vectorización con TF-IDF (max_features=1000, unigramas, min_df=5) → matriz de 997x701.
  Se redujo desde una configuración inicial de 5000 features y bigramas tras detectar
  overfitting severo (16-37 puntos de diferencia train/test en los 4 modelos).
- Se comparó el overfitting con dos métodos: train vs test (poco riguroso, ya que el
  modelo ya vio esos datos) y cross-validation (5-fold) vs test (más riguroso). Con CV,
  3 de los 4 modelos generalizaban bien (Logistic Regression 4.96 pts, Naive Bayes
  -0.09 pts, SVM 0.71 pts), y solo Random Forest optimizado con GridSearchCV seguía
  por encima del umbral (8.60 pts), además de tener el peor F1 test (0.57).
- Modelo final: ensemble por votación mayoritaria (VotingClassifier, hard voting) de
  Logistic Regression, Naive Bayes y SVM. Resultado: F1 (clase tóxico) = 0.63,
  overfitting CV vs test = 4.64 puntos (dentro del umbral). Se descarta hard voting
  soft porque SVM (LinearSVC) no calcula probabilidades por defecto.
- Modelo y vectorizador guardados en `models/` con joblib para su uso en el frontend.
- Prueba con 10 comentarios reales de YouTube (fuente: Sage Journals): el modelo
  identifica correctamente los 5 no tóxicos y 2 de los 5 tóxicos. Los que falla
  ("She is a brat.", "kick her pompous ass", "What a spoiled child") son insultos
  sin vocabulario explícitamente ofensivo, mientras que acierta en los que sí lo
  tienen ("bitch", "fuck off"). Esto indica que el modelo detecta bien la toxicidad
  explícita pero tiene más dificultad con la toxicidad implícita o sin palabrotas
  evidentes — una limitación real a tener en cuenta en la presentación.